# Melawan Overfitting: Kapasitas, Regularisasi Bobot, dan Dropout

**Buku:** *Deep Learning dengan Python* — Bab 4, Fondasi Machine Learning

Membandingkan model kecil, sedang, dan besar, lalu efek regularisasi L2 dan dropout pada kurva loss validasi IMDB.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- **Metodologi diperbaiki:** asli memakai *data uji* sebagai data validasi; kini memakai 10.000 sampel validasi dari data latih.
- Bug asli `np.randint` (tidak ada) → `np.random.default_rng().integers`.
- Satu fungsi `latih()` untuk semua varian sehingga perbandingan adil.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
import numpy as np
from keras import layers, regularizers
from keras.datasets import imdb

(data_latih, label_latih), _ = imdb.load_data(num_words=10000)

def vektorisasi(sekuens, dimensi=10000):
    hasil = np.zeros((len(sekuens), dimensi), dtype="float32")
    for i, s in enumerate(sekuens):
        hasil[i, s] = 1.0
    return hasil

x = vektorisasi(data_latih); y = np.asarray(label_latih, dtype="float32")
x_val, x_latih = x[:10000], x[10000:]
y_val, y_latih = y[:10000], y[10000:]

def latih(daftar_lapisan, epochs=20):
    model = keras.Sequential([keras.Input(shape=(10000,))] + daftar_lapisan +
                             [layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    r = model.fit(x_latih, y_latih, epochs=epochs, batch_size=512,
                  validation_data=(x_val, y_val), verbose=0)
    return r.history

## 1. Mengubah kapasitas model

In [ ]:
h_asli  = latih([layers.Dense(16, activation="relu"), layers.Dense(16, activation="relu")])
h_kecil = latih([layers.Dense(4, activation="relu"), layers.Dense(4, activation="relu")])
h_besar = latih([layers.Dense(512, activation="relu"), layers.Dense(512, activation="relu")])

In [ ]:
import matplotlib.pyplot as plt

def bandingkan(hasil, kunci="val_loss", judul="Loss validasi"):
    for nama, h in hasil.items():
        plt.plot(range(1, len(h[kunci]) + 1), h[kunci], "o-", ms=3, label=nama)
    plt.xlabel("Epoch"); plt.ylabel(kunci); plt.title(judul); plt.legend(); plt.show()

bandingkan({"Asli (16)": h_asli, "Kecil (4)": h_kecil, "Besar (512)": h_besar})
bandingkan({"Asli (16)": h_asli, "Besar (512)": h_besar}, "loss", "Loss latih")

## 2. Regularisasi bobot L2

In [ ]:
h_l2 = latih([layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(0.002)),
              layers.Dense(16, activation="relu", kernel_regularizer=regularizers.l2(0.002))])
bandingkan({"Asli": h_asli, "L2": h_l2})
# Alternatif: regularizers.l1(0.001) atau regularizers.l1_l2(l1=0.001, l2=0.001)

## 3. Dropout
Demonstrasi dengan NumPy (versi *inverted dropout*: skala dilakukan saat pelatihan):

In [ ]:
rng = np.random.default_rng(0)
keluaran = rng.random((4, 6)).astype("float32")
laju = 0.5
masker = rng.random(keluaran.shape) >= laju     # True = unit dipertahankan
keluaran_latih = keluaran * masker / (1 - laju)  # saat latih
keluaran_uji = keluaran                          # saat uji: tidak diubah
print(np.round(keluaran_latih, 2))

In [ ]:
h_dropout = latih([layers.Dense(16, activation="relu"), layers.Dropout(0.5),
                   layers.Dense(16, activation="relu"), layers.Dropout(0.5)])
bandingkan({"Asli": h_asli, "Dropout 0,5": h_dropout})

## Rangkuman
Cara utama mencegah overfitting: tambah data, kurangi kapasitas, regularisasi bobot, dan dropout.
Seluruh keputusan diambil berdasarkan **data validasi**, bukan data uji.